# Train Parasol Insurance Anomaly Detection Model

## Overview

This notebook trains an **Isolation Forest** model specifically for detecting anomalies
in the Parasol Insurance application running in a separate namespace. It collects real
or synthetic container metrics, engineers features, trains an sklearn Pipeline, and saves
a `model.pkl` ready for KServe deployment.

## Cross-Namespace Architecture

```
parasol-insurance namespace         self-healing-platform namespace
┌─────────────────────┐            ┌──────────────────────────────┐
│ ic-app (Flask)      │◄───heal────│ coordination-engine          │
│ claimdb (PostgreSQL)│            │ parasol-anomaly-detector     │
│ ServiceMonitor ─────┼───scrape──►│   (KServe InferenceService)  │
└─────────────────────┘            └──────────────────────────────┘
```

## Prerequisites

- Platform workbench running with Prometheus access
- Parasol Insurance deployed in `parasol-insurance` namespace
- `scikit-learn`, `pandas`, `numpy`, `joblib` installed

## References

- Platform notebook: `notebooks/02-anomaly-detection/01-isolation-forest-implementation.ipynb`
- ADR-002: Hybrid Deterministic-AI Self-Healing Approach
- Issue #146: Cross-namespace self-healing demo

## 1. Setup and Configuration

In [ ]:
import os
import json
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

warnings.filterwarnings('ignore')

# Configuration
PARASOL_NAMESPACE = os.environ.get('PARASOL_NAMESPACE', 'parasol-insurance')
PROMETHEUS_URL = os.environ.get(
    'PROMETHEUS_URL',
    'http://prometheus-k8s.openshift-monitoring.svc:9090',
)
MODEL_NAME = 'parasol-anomaly-detector'
MODEL_DIR = Path(f'/opt/app-root/src/models/{MODEL_NAME}')
MODEL_PATH = MODEL_DIR / 'model.pkl'
TRAINING_HOURS = int(os.environ.get('TRAINING_HOURS', '168'))  # 1 week default
DATA_SOURCE = os.environ.get('DATA_SOURCE', 'synthetic')  # synthetic | prometheus | hybrid

print(f'Parasol namespace : {PARASOL_NAMESPACE}')
print(f'Prometheus URL    : {PROMETHEUS_URL}')
print(f'Model output      : {MODEL_PATH}')
print(f'Data source       : {DATA_SOURCE}')
print(f'Training window   : {TRAINING_HOURS}h')

## 2. Data Collection

Collect container metrics from Prometheus for the `parasol-insurance` namespace.
Falls back to synthetic data if Prometheus is unreachable.

In [ ]:
def query_prometheus_range(query: str, hours: int = 168, step: str = '5m') -> pd.DataFrame:
    """Query Prometheus range API and return a DataFrame."""
    import requests

    end = datetime.utcnow()
    start = end - timedelta(hours=hours)
    params = {
        'query': query,
        'start': start.isoformat() + 'Z',
        'end': end.isoformat() + 'Z',
        'step': step,
    }
    try:
        resp = requests.get(f'{PROMETHEUS_URL}/api/v1/query_range', params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        if data['status'] != 'success' or not data['data']['result']:
            return pd.DataFrame()
        rows = []
        for series in data['data']['result']:
            for ts, val in series['values']:
                rows.append({
                    'timestamp': datetime.utcfromtimestamp(float(ts)),
                    'value': float(val),
                    **series['metric'],
                })
        return pd.DataFrame(rows)
    except Exception as e:
        print(f'  ⚠ Prometheus query failed: {e}')
        return pd.DataFrame()


def collect_prometheus_metrics() -> pd.DataFrame:
    """Collect real metrics from Prometheus for the Parasol namespace."""
    print('Collecting Prometheus metrics...')
    queries = {
        'cpu_usage': f'rate(container_cpu_usage_seconds_total{{namespace="{PARASOL_NAMESPACE}", container!=""}}[5m])',
        'memory_usage': f'container_memory_working_set_bytes{{namespace="{PARASOL_NAMESPACE}", container!=""}}',
        'memory_limit': f'container_spec_memory_limit_bytes{{namespace="{PARASOL_NAMESPACE}", container!=""}}',
        'restarts': f'increase(kube_pod_container_status_restarts_total{{namespace="{PARASOL_NAMESPACE}"}}[10m])',
        'network_rx': f'rate(container_network_receive_bytes_total{{namespace="{PARASOL_NAMESPACE}"}}[5m])',
        'network_tx': f'rate(container_network_transmit_bytes_total{{namespace="{PARASOL_NAMESPACE}"}}[5m])',
    }

    frames = {}
    for name, query in queries.items():
        df = query_prometheus_range(query, hours=TRAINING_HOURS)
        if not df.empty:
            frames[name] = df
            print(f'  ✓ {name}: {len(df)} samples')
        else:
            print(f'  ✗ {name}: no data')

    if not frames:
        print('  ⚠ No Prometheus data available')
        return pd.DataFrame()

    # Pivot into a wide feature table keyed by timestamp
    combined = None
    for name, df in frames.items():
        series = df.groupby('timestamp')['value'].mean().rename(name)
        if combined is None:
            combined = series.to_frame()
        else:
            combined = combined.join(series, how='outer')

    combined = combined.fillna(0).sort_index()
    print(f'\nCombined feature table: {combined.shape}')
    return combined


print('Data collection functions ready.')

## 3. Synthetic Data Generation

Generate realistic training data that mimics Parasol Insurance container metrics.
This is used when Prometheus data is unavailable or to augment real data.

In [ ]:
def generate_synthetic_data(n_samples: int = 5000, anomaly_fraction: float = 0.1) -> pd.DataFrame:
    """Generate synthetic container metrics mimicking Parasol Insurance workload."""
    np.random.seed(42)
    n_normal = int(n_samples * (1 - anomaly_fraction))
    n_anomaly = n_samples - n_normal

    # Normal operating ranges for Parasol Insurance
    normal = pd.DataFrame({
        'cpu_usage': np.random.normal(0.15, 0.05, n_normal).clip(0, 1),
        'memory_usage': np.random.normal(180e6, 30e6, n_normal).clip(0),      # ~180 MB
        'memory_limit': np.full(n_normal, 512e6),                              # 512 MB limit
        'restarts': np.zeros(n_normal),
        'network_rx': np.random.normal(50000, 15000, n_normal).clip(0),        # ~50 KB/s
        'network_tx': np.random.normal(30000, 10000, n_normal).clip(0),        # ~30 KB/s
        'label': 0,
    })

    # Anomalous patterns matching chaos scenarios
    anomaly_types = {
        'oom_risk': {  # memory-leak.sh scenario
            'cpu_usage': np.random.normal(0.3, 0.1, n_anomaly // 5),
            'memory_usage': np.random.normal(450e6, 40e6, n_anomaly // 5),
            'memory_limit': np.full(n_anomaly // 5, 512e6),
            'restarts': np.zeros(n_anomaly // 5),
            'network_rx': np.random.normal(50000, 15000, n_anomaly // 5),
            'network_tx': np.random.normal(30000, 10000, n_anomaly // 5),
        },
        'cpu_spike': {  # cpu-spike.sh scenario
            'cpu_usage': np.random.normal(0.85, 0.1, n_anomaly // 5),
            'memory_usage': np.random.normal(200e6, 30e6, n_anomaly // 5),
            'memory_limit': np.full(n_anomaly // 5, 512e6),
            'restarts': np.zeros(n_anomaly // 5),
            'network_rx': np.random.normal(50000, 15000, n_anomaly // 5),
            'network_tx': np.random.normal(30000, 10000, n_anomaly // 5),
        },
        'crash_loop': {  # crash-loop.sh scenario
            'cpu_usage': np.random.normal(0.05, 0.02, n_anomaly // 5),
            'memory_usage': np.random.normal(50e6, 20e6, n_anomaly // 5),
            'memory_limit': np.full(n_anomaly // 5, 512e6),
            'restarts': np.random.poisson(5, n_anomaly // 5).astype(float),
            'network_rx': np.random.normal(5000, 2000, n_anomaly // 5),
            'network_tx': np.random.normal(2000, 1000, n_anomaly // 5),
        },
        'network_issue': {  # network-partition.yaml scenario
            'cpu_usage': np.random.normal(0.5, 0.15, n_anomaly // 5),
            'memory_usage': np.random.normal(250e6, 50e6, n_anomaly // 5),
            'memory_limit': np.full(n_anomaly // 5, 512e6),
            'restarts': np.random.poisson(1, n_anomaly // 5).astype(float),
            'network_rx': np.random.normal(500, 200, n_anomaly // 5),
            'network_tx': np.random.normal(200, 100, n_anomaly // 5),
        },
        'disk_pressure': {  # disk-pressure.sh scenario
            'cpu_usage': np.random.normal(0.4, 0.1, n_anomaly // 5),
            'memory_usage': np.random.normal(350e6, 50e6, n_anomaly // 5),
            'memory_limit': np.full(n_anomaly // 5, 512e6),
            'restarts': np.random.poisson(2, n_anomaly // 5).astype(float),
            'network_rx': np.random.normal(10000, 5000, n_anomaly // 5),
            'network_tx': np.random.normal(60000, 20000, n_anomaly // 5),
        },
    }

    anomaly_frames = []
    for name, cols in anomaly_types.items():
        df = pd.DataFrame(cols)
        df['label'] = 1
        anomaly_frames.append(df)
        print(f'  Generated {len(df)} {name} anomalies')

    anomalies = pd.concat(anomaly_frames, ignore_index=True)
    data = pd.concat([normal, anomalies], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

    print(f'\nSynthetic dataset: {len(data)} samples ({n_normal} normal, {len(anomalies)} anomalous)')
    return data


print('Synthetic data generator ready.')

## 4. Collect Training Data

In [ ]:
if DATA_SOURCE == 'prometheus':
    data = collect_prometheus_metrics()
    if data.empty:
        print('\n⚠ Falling back to synthetic data')
        data = generate_synthetic_data()
    else:
        # Prometheus data has no labels -- use unsupervised mode
        data['label'] = -1  # unknown

elif DATA_SOURCE == 'hybrid':
    real_data = collect_prometheus_metrics()
    synth_data = generate_synthetic_data(n_samples=3000)
    if not real_data.empty:
        real_data['label'] = -1
        feature_cols = [c for c in synth_data.columns if c != 'label']
        for col in feature_cols:
            if col not in real_data.columns:
                real_data[col] = 0
        data = pd.concat([real_data[feature_cols + ['label']], synth_data], ignore_index=True)
        print(f'\nHybrid dataset: {len(real_data)} real + {len(synth_data)} synthetic = {len(data)} total')
    else:
        data = synth_data
        print('\n⚠ No real data; using synthetic only')

else:  # synthetic
    data = generate_synthetic_data()

FEATURE_COLS = ['cpu_usage', 'memory_usage', 'memory_limit', 'restarts', 'network_rx', 'network_tx']
print(f'\nFeature columns: {FEATURE_COLS}')
print(f'Dataset shape  : {data.shape}')
data.describe()

## 5. Feature Engineering

In [ ]:
# Derived features that are meaningful for self-healing
data['memory_utilization'] = data['memory_usage'] / data['memory_limit'].replace(0, 1)
data['network_ratio'] = data['network_tx'] / data['network_rx'].replace(0, 1)
data['cpu_memory_product'] = data['cpu_usage'] * data['memory_utilization']

FEATURE_COLS_EXTENDED = FEATURE_COLS + ['memory_utilization', 'network_ratio', 'cpu_memory_product']

X = data[FEATURE_COLS_EXTENDED].values
y = data['label'].values if 'label' in data.columns else None

print(f'Feature matrix: {X.shape}')
print(f'Features: {FEATURE_COLS_EXTENDED}')
if y is not None:
    unique, counts = np.unique(y, return_counts=True)
    for u, c in zip(unique, counts):
        lbl = {0: 'normal', 1: 'anomaly', -1: 'unlabeled'}[int(u)]
        print(f'  {lbl}: {c}')

## 6. Train Isolation Forest

In [ ]:
# Build an sklearn Pipeline (StandardScaler + IsolationForest)
# This is the same pattern used by the platform's anomaly-detector model
# so KServe can serve it with the standard sklearn server.

contamination = 0.1  # Expected anomaly fraction

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', IsolationForest(
        n_estimators=200,
        max_samples='auto',
        contamination=contamination,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    )),
])

print('Training Isolation Forest pipeline...')
pipeline.fit(X)
print('✓ Training complete')

# Predict (Isolation Forest: 1 = normal, -1 = anomaly)
predictions = pipeline.predict(X)
scores = pipeline.decision_function(X)

# Convert to binary: 0 = normal, 1 = anomaly
pred_labels = (predictions == -1).astype(int)

print(f'\nPrediction summary:')
print(f'  Normal  : {(pred_labels == 0).sum()}')
print(f'  Anomaly : {(pred_labels == 1).sum()}')
print(f'  Anomaly rate: {pred_labels.mean():.2%}')

## 7. Evaluate Model

In [ ]:
# Only evaluate if we have ground-truth labels
if y is not None and set(np.unique(y)) <= {0, 1}:
    print('Classification Report (synthetic ground truth):')
    print('=' * 55)
    print(classification_report(y, pred_labels, target_names=['Normal', 'Anomaly']))

    cm = confusion_matrix(y, pred_labels)
    print(f'Confusion Matrix:')
    print(f'  TN={cm[0,0]}  FP={cm[0,1]}')
    print(f'  FN={cm[1,0]}  TP={cm[1,1]}')

    precision = precision_score(y, pred_labels)
    recall = recall_score(y, pred_labels)
    f1 = f1_score(y, pred_labels)
    print(f'\nPrecision: {precision:.4f}')
    print(f'Recall   : {recall:.4f}')
    print(f'F1 Score : {f1:.4f}')

    # Quality gate
    assert f1 > 0.6, f'Model F1 score ({f1:.4f}) below threshold (0.6)'
    print('\n✓ Model passes quality gate (F1 > 0.6)')
else:
    print('No ground-truth labels available (Prometheus mode) -- skipping classification report.')
    print(f'Anomaly score range: [{scores.min():.4f}, {scores.max():.4f}]')
    print(f'Score mean: {scores.mean():.4f}, std: {scores.std():.4f}')

## 8. Save Model

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline, MODEL_PATH)
print(f'✓ Model saved to {MODEL_PATH}')
print(f'  File size: {MODEL_PATH.stat().st_size / 1024:.1f} KB')

# Save training metadata
metadata = {
    'model_name': MODEL_NAME,
    'model_type': 'IsolationForest',
    'training_date': datetime.utcnow().isoformat(),
    'data_source': DATA_SOURCE,
    'training_hours': TRAINING_HOURS,
    'n_samples': len(data),
    'n_features': len(FEATURE_COLS_EXTENDED),
    'features': FEATURE_COLS_EXTENDED,
    'contamination': contamination,
    'n_estimators': 200,
    'parasol_namespace': PARASOL_NAMESPACE,
    'sklearn_pipeline_steps': ['StandardScaler', 'IsolationForest'],
}

metadata_path = MODEL_DIR / 'training_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'✓ Metadata saved to {metadata_path}')

# Verify the saved model loads correctly
loaded = joblib.load(MODEL_PATH)
test_pred = loaded.predict(X[:5])
print(f'✓ Model reload verification: {test_pred}')

## 9. Summary

The trained model is saved as an sklearn Pipeline at the path above.
KServe can serve it directly using the `kserve-sklearnserver` runtime.

**Next steps:**

1. Deploy as InferenceService: the platform Helm chart creates an InferenceService
   pointing to this model path when `parasolInsurance.enabled: true`.
2. Inject a fault using `chaos/memory-leak.sh` and observe the coordination engine
   detect and remediate the anomaly in the parasol-insurance namespace.
3. Promote the model to production using `export/export-to-s3.sh`.